# QLoRA Fine-Tuning — Travel Concierge on Gemma-2B-it

Fine-tunes `google/gemma-2b-it` on a domain-specific travel Q&A dataset using
**QLoRA** (4-bit quantisation + rank-16 LoRA adapters).

The resulting adapter is saved to `./travel-concierge-adapter/` and can be
loaded by the AI agent service via the `USE_FINETUNED_MODEL=true` env flag,
with Gemini 2.5 Flash as a production fallback.

**Runtime:** Google Colab T4 GPU (free tier) — ~15 min for 100 steps.

**Requirements:**
- HuggingFace account with Gemma access accepted at https://huggingface.co/google/gemma-2b-it
- `HF_TOKEN` set in Colab Secrets (or `os.environ['HF_TOKEN']`)


In [ ]:
# Install dependencies
!pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 \
    bitsandbytes==0.43.1 datasets==2.19.0 accelerate==0.29.3 \
    huggingface_hub einops
print('Dependencies installed.')

In [ ]:
import os, torch
from huggingface_hub import login

# Authenticate — paste your HF token or use Colab Secrets
HF_TOKEN = os.getenv('HF_TOKEN', '')
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HuggingFace login successful.')
else:
    print('WARNING: HF_TOKEN not set. Set it via Colab Secrets > HF_TOKEN.')

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Travel Q&A Dataset

50 domain-specific instruction-response pairs covering flights, hotels, car
rentals, budget planning, and travel tips. Each example is formatted as a
Gemma chat-template turn.


In [ ]:
TRAVEL_QA = [
    {"question": "What are the cheapest flights from San Francisco to New York in March?",
     "answer": "For SFO→JFK in March, look for Tuesday or Wednesday departures — midweek fares typically run $180–$260 round-trip. JetBlue and Alaska Airlines consistently offer the lowest fares; set a price alert and book 6–8 weeks out."},
    {"question": "Which hotels in Paris are best for a honeymoon under $200 per night?",
     "answer": "For a romantic Paris stay under $200/night, consider Hôtel des Arts Montmartre or Hôtel Lenox Saint-Germain — both offer boutique charm, central locations, and breakfast included. Book direct for best rates."},
    {"question": "Is it cheaper to rent a car at the airport or in the city centre?",
     "answer": "City-centre rental offices are typically 15–25% cheaper than airport desks due to lower concession fees. Factor in taxi/transit cost to the off-airport location — the savings usually still hold for rentals over 3 days."},
    {"question": "What is the best time to visit Bali to avoid crowds?",
     "answer": "Visit Bali in September or October — the wet season ends, rice terraces are lush, and crowds are 40% lighter than the July–August peak. Flights and hotels drop 20–30% in this shoulder window."},
    {"question": "How far in advance should I book international flights for the best price?",
     "answer": "For international routes, the sweet spot is 2–5 months before departure. Booking more than 6 months out rarely saves money; waiting inside 3 weeks usually costs 30–40% more."},
    {"question": "What is economy plus and is it worth it?",
     "answer": "Economy Plus (or Premium Economy) adds 4–6 inches of extra legroom and priority boarding for $50–$150 extra on domestic routes. Worth it for flights over 4 hours, especially if you're tall or travelling overnight."},
    {"question": "How do I find last-minute hotel deals?",
     "answer": "Check HotelTonight or booking.com's 'Tonight' filter after 3pm local time — hotels release unsold rooms at steep discounts (30–60% off). Also try calling the hotel directly; front desk managers have discretionary pricing."},
    {"question": "What credit card is best for travel rewards?",
     "answer": "Chase Sapphire Preferred earns 3× on dining and 2× on travel with a flexible points currency redeemable across 14 airline/hotel partners. For premium perks, the Amex Platinum offers lounge access and a $200 airline credit that offsets its $695 fee."},
    {"question": "Should I buy travel insurance?",
     "answer": "Yes for international trips over 7 days or any trip costing more than $1,500. Look for policies covering trip cancellation, medical evacuation (minimum $100k), and 24/7 assistance. Skip it for cheap domestic weekend trips."},
    {"question": "What documents do I need to rent a car in Europe?",
     "answer": "You need a valid driving licence, your passport, and the credit card used for the reservation. Most European countries accept a standard US/UK licence, but Italy and some Eastern European countries require an International Driving Permit."},
    {"question": "How early should I arrive at the airport for an international flight?",
     "answer": "Arrive 3 hours before an international departure and 2 hours before domestic. If you have TSA PreCheck or Global Entry, 90 minutes before domestic flights is usually sufficient."},
    {"question": "What is the cheapest way to get from the airport to the city?",
     "answer": "Rail or metro links are almost always the cheapest option ($3–$12 vs $30–$60 for taxis). Where no rail exists, shared shuttle services halve the taxi fare. Only use taxis if you have heavy luggage and are splitting the cost with 2+ people."},
    {"question": "Can I change my flight after booking without paying a fee?",
     "answer": "Most US carriers eliminated change fees on domestic Main Cabin fares post-2020 — you pay only the fare difference. International tickets and Basic Economy fares typically still charge $100–$300 change fees. Always check the fare rules before booking."},
    {"question": "What is a resort fee and how do I avoid it?",
     "answer": "Resort fees ($20–$50/night) are mandatory charges added at check-in for amenities like the pool and wifi. Filter by 'no resort fee' on booking.com, or call the hotel and ask to have it waived for direct bookings — it works roughly 30% of the time."},
    {"question": "Is a window or aisle seat better for a long-haul flight?",
     "answer": "Aisle seats let you stretch and use the bathroom without disturbing neighbours — better for flights over 8 hours. Window seats offer a headrest wall for sleeping and no disturbances. Avoid the middle row centre seats at all costs."},
    {"question": "What is the cheapest month to fly to Europe from the US?",
     "answer": "January and February are cheapest (post-holiday, pre-spring break) — round-trip fares to London or Amsterdam average $400–$550. March and November are solid shoulder-season alternatives at $500–$700."},
    {"question": "How do hotel star ratings work?",
     "answer": "Star ratings vary by country with no universal standard. US ratings focus on amenities and service levels (3★ = comfortable, 4★ = superior, 5★ = luxury). Guest review scores on booking.com are more reliable than star ratings for predicting your actual experience."},
    {"question": "What is the best car type to rent for a road trip?",
     "answer": "For a road trip with 2–3 people and luggage, a midsize SUV (RAV4 or similar) balances fuel economy (~28 mpg highway) and cargo space. Avoid full-size SUVs unless you have 4+ passengers — they cost 30% more and get poor mileage."},
    {"question": "How do I find cheap business class flights?",
     "answer": "Use Google Flights' 'date grid' on a flexible ±3-day window, filter by Business class, and look for error fares on Secret Flying or The Flight Deal. Positioning flights (fly economy to a hub with cheaper business class) can halve the price."},
    {"question": "What amenities should I look for in a hotel?",
     "answer": "Prioritise free cancellation, wifi speed (ask for the Mbps — anything under 10 Mbps is unusable for remote work), and location walkability score. Free breakfast adds $25–$40 of real value per person per day."},
    {"question": "Is Airbnb or a hotel better for a week-long trip?",
     "answer": "For stays over 5 nights, Airbnb with a kitchen typically saves 20–40% once you factor in self-catering. Hotels win for short trips where daily housekeeping and concierge services matter. Check Airbnb's total price including cleaning fees before comparing."},
    {"question": "What are the best airlines for economy class comfort?",
     "answer": "Singapore Airlines, Qatar Airways, and ANA consistently top economy comfort rankings for seat pitch (33–34 inches), meal quality, and IFE screens. Among US carriers, Delta leads on seat comfort and on-time performance."},
    {"question": "How do I avoid paying for checked baggage?",
     "answer": "Book with airlines that include checked bags (Southwest, JetBlue Blue Extra, Alaska). Use a co-branded airline credit card — most waive the first bag fee. Pack in a carry-on: personal item + standard overhead bag covers most trips under 10 days."},
    {"question": "What is dynamic pricing in hotels and how does it affect me?",
     "answer": "Hotels update room rates hourly based on demand. Prices spike 30–60% during local events, holidays, and last-minute windows. Set a price alert on Hopper or book refundable rates early, then rebook if the price drops."},
    {"question": "Should I exchange currency before travelling or at the destination?",
     "answer": "Use a no-foreign-transaction-fee credit card (Chase Sapphire, Schwab debit) for all purchases — you get interbank rates with zero markup. Withdraw cash from local ATMs on arrival for taxis and markets; airport exchange booths charge 8–12% margins."},
    {"question": "What is the best way to search for cheap flights?",
     "answer": "Use Google Flights' Explore map to find cheapest destinations from your origin. For specific routes, check Kayak, Skyscanner, and Scott's Cheap Flights (premium). Clear cookies and use incognito mode — some sites raise prices on repeated searches."},
    {"question": "How do I upgrade to first class cheaply?",
     "answer": "Check in online exactly 24 hours before departure and look for upgrade offers in the airline app — carriers often sell unsold first-class seats for $50–$150 at check-in. Accrue miles with a co-branded card and redeem for upgrades on saver awards."},
    {"question": "What is an open-jaw flight ticket?",
     "answer": "An open-jaw ticket lets you fly into one city and out of another (e.g., fly into Rome, exit from Barcelona) — perfect for one-way road trips. It typically costs the same as a round-trip and saves you backtracking."},
    {"question": "How do hotel loyalty programs work?",
     "answer": "Loyalty programs (Marriott Bonvoy, Hilton Honors) award points per dollar spent, redeemable for free nights. Elite status (after 25–50 nights/year) unlocks room upgrades, late checkout, and bonus points. Concentrate stays at one chain to hit status faster."},
    {"question": "What is a one-way car rental and are the fees high?",
     "answer": "One-way rentals let you pick up in City A and drop off in City B. Drop fees vary wildly — $0 on popular routes (Las Vegas→LA) to $300+ on uncommon ones. Compare with driving round-trip; sometimes a round-trip is cheaper even if you don't need it."},
    {"question": "What are the safest neighbourhoods to stay in Rome?",
     "answer": "Prati (near Vatican), Trastevere, and the historic centre (around Campo de' Fiori) are safe, walkable, and close to major sights. Avoid staying near Termini station — it's convenient but has higher petty theft. Book hotels with 8.0+ guest scores."},
    {"question": "How do I find hotels near an airport for an early morning flight?",
     "answer": "Filter booking.com by distance ≤2 km from the airport and sort by price. Many airport hotels offer free shuttle service and a 4am breakfast option. Compare the cost vs an Uber from your current hotel — often the math favours staying in the city."},
    {"question": "Is it worth paying for priority boarding?",
     "answer": "Only if you have a large carry-on and the flight is full — overhead bin space fills up fast on busy routes. If you're travelling with just a personal item, skip priority boarding; you'll get on the plane 8 minutes earlier at best."},
    {"question": "What is a codeshare flight?",
     "answer": "A codeshare is when one airline sells seats on another airline's operated flight under its own flight number. You may check in with Airline A but board an Airline B plane — check the actual operating carrier for baggage policies and seat selection."},
    {"question": "How do I find the best car rental deals?",
     "answer": "Search AutoSlash.com — it monitors your reservation and automatically rebooks if a cheaper rate appears. Compare on Kayak, then cross-check the rental company's own site (direct bookings have simpler dispute resolution). Always decline the rental company's CDW if your credit card provides it."},
    {"question": "What should I pack for a 2-week trip to Southeast Asia?",
     "answer": "Pack light (carry-on only): 3–4 moisture-wicking shirts, 2 pairs of shorts, one light layer, sandals + trainers. Buy anything you forget locally for a fraction of US prices. A dry bag protects electronics during boat tours and beach days."},
    {"question": "How do flight price alerts work?",
     "answer": "Google Flights and Kayak let you track a specific route and email you when the price changes by more than $10. Set alerts 3–6 months before your target dates — you'll see the full pricing curve and can buy at the dip."},
    {"question": "What is the difference between refundable and non-refundable hotel rates?",
     "answer": "Refundable rates (typically 10–15% more) allow cancellation up to 24–48 hours before check-in for a full refund. Non-refundable rates are prepaid and forfeit the full amount on cancellation. Book refundable when plans are uncertain; switch to non-refundable once confirmed."},
    {"question": "What is the best month to visit Japan?",
     "answer": "Late March–early April (cherry blossom season) and November (autumn foliage) are peak beauty but also peak crowds. May and October offer similarly good weather with 30–40% fewer tourists and lower hotel rates. Avoid Golden Week (late April–early May) — it's Japan's busiest domestic travel period."},
    {"question": "How do airline miles expire?",
     "answer": "Most US airline miles expire after 18–24 months of account inactivity. Keep your account active by making a single qualifying transaction (even buying a $5 magazine in the SkyMall or transferring 1,000 points) every 18 months. Miles earned on co-branded credit cards usually never expire."},
    {"question": "What is a hotel resort credit and how do I use it?",
     "answer": "Resort credits ($50–$200/night) are issued at check-in for use on property — spa, dining, cabanas, or water sports. Use them early in your stay; unused credits rarely carry to checkout. Combine with a credit-card travel benefit for a near-free resort experience."},
    {"question": "Is it safe to use airport WiFi?",
     "answer": "Avoid accessing banking or sensitive accounts on airport WiFi — networks are unencrypted and trivially sniffable. Use a VPN (ProtonVPN or NordVPN free tier) for all airport and hotel WiFi sessions. Your phone's mobile data is significantly safer."},
    {"question": "How do I get a seat upgrade on a hotel room?",
     "answer": "Check in late afternoon when front desk agents have the clearest picture of availability. Mention a special occasion (honeymoon, birthday) and politely ask if any complimentary upgrades are available. Loyalty status dramatically increases upgrade odds — even Silver status helps."},
    {"question": "What is the best travel app for finding flights?",
     "answer": "Google Flights is the best free tool for price exploration and date-grid searching. Hopper accurately predicts fare changes (buy now vs wait). Scott's Cheap Flights (now Going) sends curated mistake fares directly to your inbox — the premium tier ($49/year) easily pays for itself."},
    {"question": "How do I avoid extra charges when returning a rental car?",
     "answer": "Photograph every panel and interior surface at pickup and email the photos to yourself (timestamp evidence). Return with the same fuel level as pickup. Decline optional insurance if your credit card covers CDW. Return 30 minutes before your scheduled time to allow for paperwork."},
    {"question": "What is the best seat on a long-haul flight?",
     "answer": "Use SeatGuru to check your specific aircraft. Bulkhead seats have extra legroom but no under-seat storage. Exit rows are roomy but recline is often restricted. Avoid the last row (seats don't recline) and seats adjacent to lavatories (heavy foot traffic and odour)."},
    {"question": "Can I negotiate hotel prices?",
     "answer": "Yes — call the hotel directly (not OTAs) and ask for their best available rate. For stays over 5 nights, ask for a weekly rate. During low-occupancy periods (Sunday–Thursday, off-season), discounts of 15–25% are routinely granted to callers who ask politely."},
    {"question": "What vaccinations do I need for international travel?",
     "answer": "Check the CDC Traveler's Health site for your specific destination. Yellow fever vaccination is required (not just recommended) for entry into several African and South American countries. Schedule a travel clinic appointment 6–8 weeks before departure — some vaccines require a series."},
    {"question": "How do I find a good deal on a cruise?",
     "answer": "Book cruises 6–12 months out for the best cabin selection, or within 90 days for last-minute deals (lines heavily discount to fill ships). Use CruiseCompete.com to get competing quotes from multiple agents. Repositioning cruises (one-way ocean crossings) offer 50–70% off regular per-night rates."},
    {"question": "What is a travel concierge and what can they arrange?",
     "answer": "A travel concierge handles logistics you'd otherwise spend hours on — restaurant reservations, museum timed-entry tickets, private transfers, and visa applications. Hotel concierges do this free of charge; independent concierge services charge $50–$150/hour but can access sold-out experiences."},
]

print(f'Dataset size: {len(TRAVEL_QA)} examples')

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a concise travel concierge. "
    "Give a one or two sentence actionable recommendation based on the user's question."
)

def format_gemma_chat(example):
    """Format a Q&A pair as a Gemma instruction-tuning example."""
    text = (
        f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{example['question']}<end_of_turn>\n"
        f"<start_of_turn>model\n{example['answer']}<end_of_turn>"
    )
    return {"text": text}

raw_dataset = Dataset.from_list(TRAVEL_QA)
dataset = raw_dataset.map(format_gemma_chat)

print('Sample formatted example:')
print(dataset[0]['text'])

## 2. Load Gemma-2B-it with 4-bit Quantisation (QLoRA)


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

BASE_MODEL = "google/gemma-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",           # NormalFloat4 — best quality for weights
    bnb_4bit_compute_dtype=torch.float16, # compute in fp16 for speed
    bnb_4bit_use_double_quant=True,       # double quantisation saves ~0.4 bits/param
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",  # flash-attn not available on all Colab GPUs
)
model.config.use_cache = False
model.config.pretraining_tp = 1

print(f'Model loaded: {BASE_MODEL}')
print(f'Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 3. Apply LoRA Adapters (PEFT)

Rank-16 LoRA adapters on the query and value projection matrices.  
Only ~1% of model parameters are trained — full model stays frozen in 4-bit.


In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                        # rank — higher = more capacity, more params
    lora_alpha=32,               # scaling factor (alpha/r = 2 is standard)
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 4. Train with SFTTrainer


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "./travel-concierge-adapter"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,       # effective batch size = 16
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    max_steps=150,                       # ~15 min on Colab T4; remove for full training
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    tokenizer=tokenizer,
    args=training_args,
    peft_config=lora_config,
)

print('Starting fine-tuning...')
trainer.train()
print('Training complete.')

In [ ]:
# Save the LoRA adapter weights
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Adapter saved to {OUTPUT_DIR}')

## 5. Test Inference with Fine-Tuned Adapter


In [ ]:
from peft import PeftModel
from transformers import pipeline

# Reload base model and merge adapter for inference
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(base, OUTPUT_DIR)
ft_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)

pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    max_new_tokens=200,
    temperature=0.4,
    do_sample=True,
)

def ask(question: str) -> str:
    prompt = (
        f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{question}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )
    out = pipe(prompt)
    generated = out[0]["generated_text"]
    return generated[len(prompt):].strip()

# Run test queries
test_questions = [
    "What is the cheapest time to fly to Tokyo from Los Angeles?",
    "Should I book a hotel or Airbnb for a 10-day Europe trip?",
    "How do I avoid paying extra fees when renting a car?",
]

for q in test_questions:
    print(f'Q: {q}')
    print(f'A: {ask(q)}')
    print()

## 6. Deploy the Adapter in the AI Agent Service

Copy the adapter directory to the project and enable it via environment variable:

```bash
# Copy adapter to project
cp -r ./travel-concierge-adapter/ ../notebooks/adapter/

# Enable in ai-agent/.env
USE_FINETUNED_MODEL=true
FINETUNED_MODEL_PATH=notebooks/adapter
```

The agent service will load the fine-tuned adapter on startup and use it for
all `/chat` requests. If the adapter is unavailable or inference fails, the
service automatically falls back to **Gemini 2.5 Flash**.

### Architecture with fine-tuned model

```
POST /chat
  │
  ├─ RAG retrieval + multi-signal ranking
  │
  ├─ Fine-tuned Gemma-2B-it (QLoRA adapter)  ← primary
  │   └─ if unavailable or fails
  └─ Gemini 2.5 Flash                        ← fallback
```
